In [ ]:
from mnist import Net, train, test
from hyptorch.pmath import dist0

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0)

state_dict_path = "checkpoints/mnist_cnn_7.pt"
args_path = "mnist_args.json"
assert os.path.isfile(state_dict_path)
assert os.path.isfile(args_path)

data_path = "../mnist_data"
if os.getenv("GLOBALSCRATCH") is not None:
    data_path = os.path.join(os.getenv("GLOBALSCRATCH"), "mnist_data")

omni_path = os.path.join(data_path, "omniglot")

In [ ]:
save_imgs = True
if save_imgs:
    os.makedirs("exp_images", exist_ok=True)

In [ ]:
with open(args_path, "r") as f:
    args = json.load(f)
args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

In [ ]:
model = Net(args).to(device)

state_dict = torch.load(state_dict_path, map_location=device)
model.load_state_dict(state_dict)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
train_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
test(None, model, device, test_MNIST_loader)

In [ ]:
test_Omniglot_loader = torch.utils.data.DataLoader(
    datasets.Omniglot(
        omni_path,
        background=False,
        download=True,
        transform=transforms.Compose(
            [
                transforms.Resize((28, 28)), 
                transforms.ToTensor(),
                transforms.Lambda(lambda image: 1.0 - image),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
omni_data, omni_target = next(iter(test_Omniglot_loader))
omni_data, omni_target = omni_data.to(device), omni_target.to(device)

num_samples = 20
omni_sample_data = omni_data.detach().cpu().numpy()[:num_samples]
omni_sample_target = omni_target.detach().cpu().numpy()[:num_samples]
omni_sample_embeddings = model.embed(omni_data)[:num_samples]

In [ ]:
data, target = next(iter(test_MNIST_loader))
data, target = data.to(device), target.to(device)
output = model(data)

num_samples = 40
sample_data = data.detach().cpu().numpy()[:num_samples]
sample_target = target.detach().cpu().numpy()[:num_samples]
sample_embeddings = model.embed(data)[:num_samples]

In [ ]:
fig, ax = plt.subplots(10//2, 2, figsize=(6, 8))

for i in range(10):
    ax[i//2, i%2].imshow(sample_data[i, 0], aspect="equal", cmap="gray")
    ax[i//2, i%2].axis("off")

vals = np.exp(model(data).detach().cpu().numpy()[:10])
sample_pred = np.argmax(vals, axis=1)
sample_conf = np.max(vals, axis=1)
for i in range(10):
    ax[i//2, i%2].set_title(f"Pred: {sample_pred[i]}\nConf: {sample_conf[i]:.2f}")

plt.tight_layout()
plt.show()

### Figure 1 from the paper

In [ ]:
np_sample_embeddings = sample_embeddings.detach().cpu().numpy()
np_omni_sample_embeddings = omni_sample_embeddings.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))

circle = plt.Circle((0, 0), 1, color="black", fill=False)
ax.add_artist(circle)

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (x, y) in enumerate(np_sample_embeddings):
    img = sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color=colors[sample_target[i]], s=50)

for i, (x, y) in enumerate(np_omni_sample_embeddings):
    img = omni_sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color="black", s=50, marker="x")

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

plt.axis("off")
if save_imgs:
    plt.savefig("exp_images/embedding_plot.png")
else:
    plt.show()

### Compare distributions

In [ ]:
def collect_dist0(model, loader, device):
    model.eval()
    distances = []

    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            embedding = model.embed(data)
            dist = dist0(embedding, c=model.tp.c).view(-1)
            distances.extend(dist.cpu().numpy())

    return np.array(distances)

In [ ]:
MNIST_dist = collect_dist0(model, test_MNIST_loader, device)
omni_dist = collect_dist0(model, test_Omniglot_loader, device)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6)) 

ax.hist(MNIST_dist, bins=50, alpha=0.4, color="red", density=True, label="MNIST")
ax.hist(omni_dist, bins=50, alpha=0.4, color="blue", density=True, label="Omniglot")
ax.set_xlabel("Distance to 0")
ax.set_ylabel("Density")
ax.set_title("Histogram of Distances to 0 (n=2)")
ax.legend()

plt.grid(alpha=0.3)
if save_imgs:
    plt.savefig("exp_images/mnist_omni_hist_n2.png")
else:
    plt.show()

In [ ]:
from scipy.stats import ks_2samp

ks_stat, ks_p_value = ks_2samp(MNIST_dist, omni_dist)
print(f"KS Statistic: {ks_stat}\nKS p-value: {ks_p_value}")

In [ ]:
class EuclideanNet(nn.Module):
    def __init__(self, args):
        super(EuclideanNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5, 1)
        self.conv2 = nn.Conv2d(20, 50, 5, 1)
        self.fc1 = nn.Linear(4 * 4 * 50, 500)
        self.fc2 = nn.Linear(500, args.dim)
        self.classifier = nn.Linear(args.dim, 10)

    def embed(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 4 * 4 * 50)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def forward(self, x):
        x = self.embed(x)
        return F.log_softmax(self.classifier(x), dim=-1)

In [ ]:
euclidean_net = EuclideanNet(args).to(device)

optimizer = optim.Adam(euclidean_net.parameters(), lr=args.lr)

os.makedirs("checkpoints", exist_ok=True)
for epoch in range(1, args.epochs + 1):
    train(args, euclidean_net, device, train_MNIST_loader, optimizer, epoch)
    test(args, euclidean_net, device, test_MNIST_loader)
    
    torch.save(euclidean_net.state_dict(), f"checkpoints/mnist_euc_cnn_{epoch}.pt")

In [ ]:
state_dict = torch.load("checkpoints/mnist_euc_cnn_3.pt", map_location=device)
euclidean_net.load_state_dict(state_dict)

In [ ]:
def euc_pmax(model, loader, device):
    model.eval()
    pmaxes = []

    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            pmax = model(data).exp().max(dim=1)[0]
            pmaxes.extend(pmax.cpu().numpy())

    return np.array(pmaxes)


In [ ]:
MNIST_pmax = euc_pmax(model, test_MNIST_loader, device)
omni_pmax = euc_pmax(model, test_Omniglot_loader, device)

In [ ]:
ks_stat, ks_p_value = ks_2samp(MNIST_pmax, omni_pmax)
print(f"KS Statistic: {ks_stat}\nKS p-value: {ks_p_value}")